### NYC Info Export Pipeline

This notebook packages the trained NYC 311 model for deployment, tests a sample prediction, and creates Tableau-ready output files.

**What this notebook does**
1. Sets project paths
2. Loads the cleaned NYC 311 dataset
3. Builds a ZIP-to-latitude/longitude lookup table
4. Defines a lightweight input-preparation function
5. Loads the saved preprocessor and trained Random Forest model
6. Combines them into one deployable inference pipeline
7. Tests a sample prediction
8. Batch-scores the cleaned dataset for Tableau
9. Exports model and dashboard artifacts

**Targets**
- `resolution_in_wk = 1` means the complaint took **more than 7 days** to close
- `resolution_in_wk = 0` means the complaint closed **within 7 days**

In [1]:
# Import libraries
from pathlib import Path
import joblib
import pandas as pd
import sys

In [2]:
# Folder paths
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Load data

This notebook uses the cleaned dataset from Notebook 1. It serves two purposes:
- input for batch scoring
- source data for a ZIP-to-latitude/longitude lookup table

In [3]:
batch_input = pd.read_parquet(DATA_DIR / "01_nyc_info_preprocessing.parquet").copy()

print(batch_input.shape)
batch_input.head()

(263110, 14)


,unique_key,agency,complaint_type,descriptor,incident_zip,borough,latitude,longitude,location_type,resolution_time_days,resolution_in_wk,complaint_hr,complaint_day,complaint_month
0,68336154,DOT,Street Condition,Pothole,11420,queens,40.678150,-73.831082,unknown,0.000000,0,1,0,3
1,68332725,NYPD,Noise - Residential,Banging/Pounding,11104,queens,40.742897,-73.925399,Residential Building/House,0.010498,0,1,0,3
2,68338535,NYPD,Blocked Driveway,Partial Access,11429,queens,40.715043,-73.737347,Street/Sidewalk,0.016169,0,1,0,3
3,68339824,NYPD,Noise - Residential,Banging/Pounding,10462,bronx,40.838182,-73.859103,Residential Building/House,0.018588,0,1,0,3
4,68335021,NYPD,Noise - Residential,Banging/Pounding,10462,bronx,40.838182,-73.859103,Residential Building/House,0.022338,0,1,0,3


#### Build ZIP-to-coordinate lookup

End users are much more likely to know a ZIP code than a latitude/longitude pair.  
To make the deployed model easier to use, this notebook converts `incident_zip` into representative coordinates using the median latitude and longitude observed in the cleaned dataset.

In [4]:
zip_lookup = (
    batch_input
    .dropna(subset=["incident_zip", "latitude", "longitude"])
    .assign(incident_zip=lambda df: df["incident_zip"].astype(str).str.strip())
    .groupby("incident_zip", as_index=False)[["latitude", "longitude"]]
    .median()
)

zip_lookup.to_csv(MODELS_DIR / "zip_lat_lon_lookup.csv", index=False)

print(zip_lookup.shape)
zip_lookup.head()

(195, 3)


,incident_zip,latitude,longitude
0,10001,40.749520,-73.995297
1,10002,40.717925,-73.988946
2,10003,40.732977,-73.988522
3,10004,40.704466,-74.012927
4,10005,40.705694,-74.008681


#### Load saved model artifacts

Notebook 3 saves the preprocessor and Notebook 4 saves the final Random Forest model.  
This notebook loads both artifacts and combines them into one deployment-ready pipeline.

In [5]:
preprocessor = joblib.load(MODELS_DIR / "preprocessor.joblib")
model = joblib.load(MODELS_DIR / "nyc_info_random_forest_model.joblib")

print("Loaded preprocessor and model successfully.")

Loaded preprocessor and model successfully.
